In [ ]:
pip install openai python-dotenv pandas tqdm

In [ ]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

project_root = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\FINAL\FINAL_ARG_prompting")

raw_text_root = project_root / "raw_texts"
llm_output_root = project_root / "llm_outputs"
llm_output_root.mkdir(exist_ok=True)


final_teds = ["TED_001", "TED_002", "TED_004", "TED_005", "TED_006", "TED_007", "TED_008", "TED_010", "TED_011", "TED_012", "TED_015", "TED_016", "TED_017", "TED_018", "TED_019", "TED_020"]
#optimization teds = ["TED_003", "TED_009", "TED_013", "TED_14"]

def read_ted_text(document_id):
    with open(raw_text_root / f"{document_id}.txt", "r", encoding="utf-8") as f:
        return f.read()

# set up

In [ ]:
client = OpenAI(
    api_key=# API KEY HERE,
)

#  models

In [ ]:
models = [
    "mistralai/mistral-small-3.2-24b-instruct",
    "qwen/qwen3.6-27b", 
    "google/gemma-4-31b-it"
]

model_file_names = {
    "mistralai/mistral-small-3.2-24b-instruct": "mistral-small-3.2-24b-instruct",
    "qwen/qwen3.6-27b": "qwen3.6-27b",
    "google/gemma-4-31b-it": "gemma-4-31b-it"
}


# read ted text

In [ ]:
def read_ted_text(document_id):
    path = raw_text_root / f"{document_id}.txt"

    with open(path, "r", encoding="utf-8") as f:
        return f.read()


# FINAL ARGUMENTATION PROMPT: used on the other 16 ted talks, and the examples used are from developmental teds

In [ ]:
system_prompt_argumentation_final = """
You are an expert annotator trained in argumentation mining.
Your task is to annotate argumentative components in TED Talk transcripts.
Return ONLY valid JSON.
Do not explain your reasoning.
"""

def build_argumentation_prompt_final(ted_text):
    return f"""
Identify all argumentation elements in the following TED Talk transcripts.
An argument is made of 3 argument components listed below.

Use ONLY these labels based on their definitions:

- MC-EXP: The major claim represents the central message or overarching conclusion of a text. The major claim represents the stance of the author about the essay topic. It is also called a thesis statement and frequently indicated by opinion expressions like "From my point of view...", "In my opinion...", "I strongly believe that...", etc. Usually, the major claim is present in the introduction or conclusion of an essay or in both. 

- C-EXP: Claims represent intermediate statements that contribute to the development of the major claim. They are typically evaluative or interpretive in nature and express a viewpoint that requires justification. 

- P-EXP: Premises provide the reasoning or evidence that supports a claim. They play a crucial role in establishing the plausibility of an argument. Premises can also proceed after indicators such as "because...", "for example...", "moreover...", etc.


IMPORTANT:
Most sentences in TED Talks are NOT argumentative.

Do NOT annotate:

- storytelling,
- greetings,
- jokes,
- rhetorical questions,
- transitions,
- descriptions,
- facts,
- background information,
- emotional language,
- personal anecdotes unless they directly support a claim.

Annotate ONLY spans that clearly function as argumentation.

RULES:

Copy EXACT text spans.
Select the MINIMAL meaningful span.
Do NOT paraphrase.
Do NOT invent spans.
Do NOT annotate weak or uncertain cases.
If unsure, do NOT annotate.
Precision is more important than recall.

Example 1:
Text:
"I teach chemistry."

Output:
{{
"annotations": []
}}

Example 2:
Text: 
"And they suppress institutions and destroy them so that they don't have to be held accountable. So do you have a heavily militarized country? And this leads to point four, what I call human cruelty."

Output:
{{
  "annotations": [
    {{
      "label": "C-EXP",
      "text": "they suppress institutions and destroy them so that they don't have to be held accountable"
    }},
    {{
      "label": "P-EXP",
      "text": "this leads to point four, what I call human cruelty"
    }}
  ]
}}

Example 3:
Text:
"Well, I believe that if we want to change what our cities look like, then we really have to change the decision-making processes that have given us the results that we have right now. We need a participation revolution, and we need it fast. The idea of voting as our only exercise in citizenship does not make sense anymore. People are tired of only being treated as empowered individuals every few years when it's time to delegate that power to someone else."

Output:
{{
  "annotations": [
    {{
      "label": "MC-EXP",
      "text": "if we want to change what our cities look like, then we really have to change the decision-making processes that have given us the results that we have right now"
    }},
    {{
      "label": "C-EXP",
      "text": "We need a participation revolution"
    }}
    {{
      "label": "C-EXP",
      "text": "we need it fast"
    }}
    {{
      "label": "C-EXP",
      "text": "The idea of voting as our only exercise in citizenship does not make sense anymore"
    }}
    {{
      "label": "P-EXP",
      "text": "People are tired of only being treated as empowered individuals every few years when it's time to delegate that power to someone else"
    }}
  ]
}}

Example 4:
"Thank you very much."

Output:
{{
"annotations": []
}}

Example 5:
Text:
"I blog and thousands of people read it."

Output:
{{
"annotations": []
}}


OUTPUT FORMAT:
{{
  "annotations": [
    {{
      "label": "C-EXP",
      "text": "exact copied span"
    }}
  ]
}}


TED Talk:
\"\"\"
{ted_text}
\"\"\"
"""


# clean JSON output

In [ ]:
def clean_llm_json_output(output):
    output = output.strip()

    output = re.sub(r"```json", "", output, flags=re.IGNORECASE)
    output = re.sub(r"```", "", output)

    start = output.find("{")
    end = output.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output.")

    output = output[start:end+1]

    parsed = json.loads(output)

    return json.dumps(parsed, ensure_ascii=False, indent=2)

# call openrouter

In [ ]:
def run_openrouter_prompt(model, system_prompt, user_prompt, temperature=0):
    kwargs = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature
    }

    # Sara said this may help for Gemma 4
    if model == "google/gemma-4-31b-it":
        kwargs["extra_body"] = {"reasoning": {"enabled": True}}

    completion = client.chat.completions.create(**kwargs)
    return completion.choices[0].message.content

# run all MODELS + ALL 16 TED TALKS


In [ ]:
prompt_version = "final"          
task_type = "argumentation"

def run_one_job(document_id, model):
    ted_text = read_ted_text(document_id)

    safe_model_name = model_file_names[model]

    output_filename = (
        f"{document_id}_{safe_model_name}_"
        f"prompt-{prompt_version}_{task_type}.json"
    )

    output_path = llm_output_root / output_filename
    raw_error_path = llm_output_root / output_filename.replace(".json", "_RAW_ERROR.txt")

    raw_output = run_openrouter_prompt(
        model=model,
        system_prompt=system_prompt_argumentation_final,
        user_prompt=build_argumentation_prompt_final(ted_text),
        temperature=0
    )

    try:
        cleaned_output = clean_llm_json_output(raw_output)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(cleaned_output)

        status = "success"

    except Exception as e:
        with open(raw_error_path, "w", encoding="utf-8") as f:
            f.write(raw_output)

        status = f"json_cleaning_error: {e}"

    return {
        "document": document_id,
        "model": safe_model_name,
        "prompt_version": prompt_version,
        "task_type": task_type,
        "output_file": str(output_path),
        "status": status
    }


jobs = []

for document_id in final_teds:
    for model in models:
        jobs.append((document_id, model))

run_log = []

with ThreadPoolExecutor(max_workers=1) as executor:
    futures = [
        executor.submit(run_one_job, document_id, model)
        for document_id, model in jobs
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            run_log.append(future.result())
        except Exception as e:
            run_log.append({
                "document": None,
                "model": None,
                "prompt_version": prompt_version,
                "task_type": task_type,
                "output_file": None,
                "status": f"api_error: {e}"
            })

run_log_df = pd.DataFrame(run_log)
run_log_df

### Check errors if needed

In [ ]:
failed = run_log_df[
    run_log_df["status"].str.contains("error", case=False, na=False)
]

for i, row in failed.iterrows():
    print("\n====================")
    print("MODEL:", row["model"])
    print("STATUS:")
    print(row["status"])

In [ ]:
run_log_path = llm_output_root / f"run_log_prompt-{prompt_version}_{task_type}.csv"

run_log_df.to_csv(run_log_path, index=False)

print("Saved run log to:", run_log_path)